# Lezione 4 — Pratica: Pie, Box, Scatter e Bubble plot

Due esercizi basati sul materiale della lezione. Ogni esercizio ha:

1. Consegna + suggerimenti.
2. Una cella di codice con `# TODO` da completare.
3. Una soluzione a comparsa per confrontare.

Usiamo lo stesso dataset (`Canada.csv`) del notebook di teoria.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('ggplot')

df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')
df_can.set_index('Country', inplace=True)

years = list(map(str, range(1980, 2014)))

print('Dataset caricato:', df_can.shape)
df_can.head()

## Esercizio 1 — Pie chart: proporzione di immigrati per continente nel 2013

Crea un pie chart che mostri la **proporzione di immigrati per continente** corrispondente solo all'anno **2013** (non il totale 1980-2013). Usa `explode` per evidenziare i **tre continenti con il contributo minore**.

**Suggerimenti:**
- Raggruppa `df_can` per `'Continent'` con `.groupby('Continent').sum(numeric_only=True)` per ottenere `df_continents`.
- La colonna di interesse non è più `'Total'`, ma la colonna dell'anno `'2013'`.
- Ordina i continenti per il valore del 2013 (`.sort_values()`) per sapere quali sono i tre con il contributo minore e così costruire la lista `explode_list` (un valore per continente, nello stesso ordine dell'indice del DataFrame che disegni).
- Usa `autopct='%1.1f%%'`, `startangle=90` e `plt.axis('equal')` come nella teoria.

In [ ]:
# TODO: raggruppa df_can per 'Continent' sommando i valori numerici -> df_continents

# TODO: identifica i tre continenti con valore minore nella colonna '2013'

# TODO: costruisci explode_list (un valore per continente: es. 0.1 per i tre minori, 0 per il resto)
# rispettando l'ORDINE di df_continents.index

# TODO: disegna il pie chart usando df_continents['2013'] con autopct, startangle, explode e legenda

# TODO: plt.axis('equal') e plt.show()


<details>
<summary>💡 Vedi soluzione</summary>

```python
# raggruppare per continente, sommando tutte le colonne numeriche (include la colonna '2013')
df_continents = df_can.groupby('Continent').sum(numeric_only=True)

# ordinare i continenti per il valore del 2013, dal minore al maggiore
orden_2013 = df_continents['2013'].sort_values()

# i tre continenti con contributo minore nel 2013
menores = set(orden_2013.index[:3])

# costruire explode_list rispettando l'ordine reale di df_continents.index
explode_list = [0.1 if continente in menores else 0 for continente in df_continents.index]

df_continents['2013'].plot(kind='pie',
                            figsize=(10, 6),
                            autopct='%1.1f%%',
                            startangle=90,
                            shadow=True,
                            pctdistance=1.12,
                            explode=explode_list)

plt.title('Immigrazione in Canada per continente (2013)', y=1.12, fontsize=15)
plt.axis('equal')
plt.legend(labels=df_continents.index, loc='upper left', fontsize=8)
plt.show()
```
</details>

## Esercizio 2 — Bubble plot: Cina vs. India (normalizzato)

Costruisci un bubble plot che confronti l'immigrazione da **Cina** e **India** (1980-2013), usando la dimensione della bolla per rappresentare la grandezza di ogni anno. Ogni paese deve essere **normalizzato separatamente** (feature scaling nell'intervallo [0, 1]) prima di creare il grafico, come fatto nella teoria con Brasile e Argentina.

**Suggerimenti:**
- Parti da `df_can_t = df_can[years].transpose()`, converti l'indice in intero e ripristinalo come colonna `'Year'` (come nella teoria).
- Normalizza ogni serie con la formula `(x - x.min()) / (x.max() - x.min())`.
- Scala il risultato normalizzato per il parametro `s`, ad esempio `norm * 2000 + 10`, affinché nessun punto sia invisibile.
- Disegna prima la Cina con `df_can_t.plot(kind='scatter', ...)`, salva l'asse restituito (`ax0`), e richiama `.plot()` per l'India passando `ax=ax0` affinché entrambi i paesi condividano gli stessi assi.

In [ ]:
# TODO: costruisci df_can_t = df_can[years].transpose(), con indice intero chiamato 'Year' ripristinato come colonna

# TODO: normalizza 'China' -> norm_china, e 'India' -> norm_india, nell'intervallo [0, 1]

# TODO: disegna lo scatter della Cina con s=norm_china * 2000 + 10, salva l'asse in ax0

# TODO: disegna lo scatter dell'India con s=norm_india * 2000 + 10, ax=ax0

# TODO: aggiungi titolo, etichetta dell'asse y e legenda ['China', 'India']; plt.show()


<details>
<summary>💡 Vedi soluzione</summary>

```python
# trasporre per avere gli anni come righe
df_can_t = df_can[years].transpose()
df_can_t.index = map(int, df_can_t.index)
df_can_t.index.name = 'Year'
df_can_t.reset_index(inplace=True)

# normalizzare Cina e India separatamente (feature scaling a [0, 1])
norm_china = (df_can_t['China'] - df_can_t['China'].min()) / (df_can_t['China'].max() - df_can_t['China'].min())
norm_india = (df_can_t['India'] - df_can_t['India'].min()) / (df_can_t['India'].max() - df_can_t['India'].min())

# Cina
ax0 = df_can_t.plot(kind='scatter',
                     x='Year',
                     y='China',
                     figsize=(14, 8),
                     alpha=0.5,
                     color='green',
                     s=norm_china * 2000 + 10,
                     xlim=(1975, 2015))

# India, sugli stessi assi della Cina
ax1 = df_can_t.plot(kind='scatter',
                     x='Year',
                     y='India',
                     alpha=0.5,
                     color='blue',
                     s=norm_india * 2000 + 10,
                     ax=ax0)

ax0.set_ylabel('Numero di immigrati')
ax0.set_title('Immigrazione da Cina e India verso il Canada (1980-2013)')
ax0.legend(['China', 'India'], loc='upper left', fontsize='x-large')

plt.show()
```
</details>